# Phase 2A 재판독 영상 선정 파이프라인

이 노트북은 `plan_phase2.md`의 **재판독 영상 선정 단계**를 그대로 구현한다. 목적은 통계 결과를 미리 만드는 것이 아니라, 기존 라벨과 전체 시퀀스에서 재판독할 영상을 선정하고 환자별 mini-sequence, hidden duplicate 반복 블록, 실제 재판독 목록과 빈 반환 양식을 만드는 것이다.

| 노트북 섹션 | 대응 계획서 |
|---|---|
| 1. 입력 정규화 | 3.2, 3.3, 3.6 |
| 2. 적격성·중첩·분모 | 3.2, 6.1 |
| 3. 표집량 명시 | 3.4, 6.1 |
| 4. 대조군 확률표집 | 3.2, 3.4, 4.1 |
| 5. mini-sequence 구성 | 3.4 |
| 6. 재판독 전 감사 | 3.3, 3.4, 4.1, 6.1 |
| 7. hidden duplicate | 3.5, 4.4, 6.1 |
| 8. 재판독 제시 목록 | 3.1, 3.3, 10 |
| 9. 산출물 저장 | 10 |

표집량과 hidden duplicate 수는 자동으로 결정하지 않는다. 섹션 2의 실제 count를 확인한 뒤 섹션 3에서 정수로 명시한다.


## 섹션 1. 입력 정규화

**무엇을 하는가:** 원본 CSV를 한 영상당 한 행, RT·RB·LT·LB 네 ROI 열을 가진 형태로 정규화한다. wide와 ROI별 long 포맷을 모두 지원한다.

**입력:** `unified_labels.csv`, cohort_column 대신 uid의 앞 두 글자를 사용한다

**출력:** `df` — `patient_id`, `seq`, `year`, `RT`, `RB`, `LT`, `LB`, `seq_position`, `image_key`를 가진 표.



In [4]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display

ROIS = ["RT", "RB", "LT", "LB"]
VALID_GRADES = {0, 1, 2, 3, 4}
RANDOM_SEED = 20260803

SOURCE_CSV = Path("/shared/home/mai/JeongGeon/Private/CXR/unified_labels.csv")
OUTPUT_DIR = SOURCE_CSV.parent / "Result" / "Phase2A"
SOURCE_IMAGE_COLUMN = None      # 예: "image_path". 없으면 None

raw = pd.read_csv(SOURCE_CSV, encoding="utf-8-sig")
raw.columns = [str(c).strip() for c in raw.columns]
lower = {c.casefold(): c for c in raw.columns}

def find_col(names):
    return next((lower[n.casefold()] for n in names if n.casefold() in lower), None)

patient_col = find_col(["patient_id", "patientid", "patient", "pid"])
seq_col = find_col(["seq", "sequence", "frame", "image_seq", "order"])
uid_col = find_col(["uid"])

if patient_col is None or seq_col is None:
    raise ValueError("patient_id와 seq 열이 필요합니다.")

if uid_col is None:
    raise ValueError("코호트 판별에 사용할 uid 열이 필요합니다.")

raw = raw.rename(
    columns={
        patient_col: "patient_id",
        seq_col: "seq",
        uid_col: "uid",
    }
)

raw["patient_id"] = raw["patient_id"].astype(str).str.strip()
raw["uid"] = raw["uid"].astype(str).str.strip()
raw["seq"] = pd.to_numeric(raw["seq"], errors="coerce")

# uid 앞 두 글자로 코호트(year) 생성
uid_prefix = raw["uid"].str[:2]
raw["year"] = uid_prefix.map({
    "24": 2024,
    "26": 2026,
})

wide_cols = {roi: lower.get(roi.casefold()) for roi in ROIS}
roi_col = find_col(["roi", "region", "quadrant"])
grade_col = find_col(["grade", "label", "severity"])

if all(wide_cols.values()):
    df = raw.rename(columns={wide_cols[r]: r for r in ROIS}).copy()

    if df.duplicated(["patient_id", "seq"]).any():
        raise ValueError("wide 포맷에서 patient_id+seq 중복이 있습니다.")

elif roi_col is not None and grade_col is not None:
    long = raw.rename(columns={roi_col: "ROI", grade_col: "grade"}).copy()
    long["ROI"] = long["ROI"].astype(str).str.upper().str.strip()
    long = long[long["ROI"].isin(ROIS)].copy()

    if long.duplicated(["patient_id", "seq", "ROI"]).any():
        raise ValueError("long 포맷에서 patient_id+seq+ROI 중복이 있습니다.")

    meta_cols = ["patient_id", "seq", "uid", "year"]

    if SOURCE_IMAGE_COLUMN and SOURCE_IMAGE_COLUMN in long.columns:
        meta_cols.append(SOURCE_IMAGE_COLUMN)

    meta = long[meta_cols].drop_duplicates(["patient_id", "seq"])
    wide = (
        long
        .pivot(
            index=["patient_id", "seq"],
            columns="ROI",
            values="grade",
        )
        .reset_index()
    )

    df = meta.merge(
        wide,
        on=["patient_id", "seq"],
        how="outer",
        validate="one_to_one",
    )

else:
    raise ValueError(
        "wide 포맷은 RT/RB/LT/LB 열, "
        "long 포맷은 ROI와 grade 열이 필요합니다."
    )

for col in ["seq", "year", *ROIS]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

checks = {
    "rows": len(df),
    "patients": df.patient_id.nunique(),
    "missing_patient_id": int(df.patient_id.eq("").sum()),
    "missing_uid": int(df.uid.eq("").sum()),
    "missing_seq": int(df.seq.isna().sum()),
    "invalid_year": int((~df.year.isin([2024, 2026])).sum()),
    "duplicate_patient_seq": int(df.duplicated(["patient_id", "seq"]).sum()),
}

for roi in ROIS:
    checks[f"missing_{roi}"] = int(df[roi].isna().sum())
    checks[f"invalid_{roi}"] = int(
        (~df[roi].isin(VALID_GRADES) & df[roi].notna()).sum()
    )

display(pd.Series(checks, name="value").to_frame())

if any(v for k, v in checks.items() if k not in {"rows", "patients"}):
    raise ValueError(
        "입력 검증표의 결측·중복·등급·코호트 오류를 확인하세요."
    )

df["seq"] = df["seq"].astype(int)
df["year"] = df["year"].astype(int)

for roi in ROIS:
    df[roi] = df[roi].astype(int)

df = df.sort_values(["patient_id", "seq"]).reset_index(drop=True)

df["seq_position"] = df.groupby("patient_id").cumcount() + 1
df["patient_n_images"] = df.groupby("patient_id")["seq"].transform("size")
df["image_key"] = (
    df["patient_id"].astype(str)
    + "|"
    + df["seq"].astype(str)
)

if SOURCE_IMAGE_COLUMN and SOURCE_IMAGE_COLUMN in df.columns:
    df["source_image"] = df[SOURCE_IMAGE_COLUMN]
else:
    df["source_image"] = pd.NA

,value
rows,1305
patients,114
missing_patient_id,0
missing_uid,0
missing_seq,0
invalid_year,0
duplicate_patient_seq,0
missing_RT,0
invalid_RT,0
missing_RB,0


## 섹션 2. 대상군 적격성, 환자 중첩 구조와 분모 재집계

**무엇을 하는가:** 표적군과 세 대조군의 후보 모집단을 먼저 센다. 하위 경계 대조군에서 새로 표집할 대상은 RB 기존 등급 2이며, 기존 등급 3은 표적군 전수에서 재사용한다.

**출력:** 환자별 적격성, 2024년 대상군 중첩표, RB 기존 등급별 영상·환자 수, 전체 모집단의 ROI별 3·4 분모.

이 결과를 확인한 뒤 다음 섹션에서 실제 대조군 표집량과 hidden duplicate 수를 명시한다.


In [5]:
patient = df.groupby(["year", "patient_id"]).agg(
    n_images=("image_key", "size"),
    has_RB0=("RB", lambda s: (s == 0).any()),
    has_RB1=("RB", lambda s: (s == 1).any()),
    has_RB2=("RB", lambda s: (s == 2).any()),
    has_RB3=("RB", lambda s: (s == 3).any()),
    has_RB4=("RB", lambda s: (s == 4).any()),
).reset_index()

patient["target_eligible"] = patient.year.eq(2024) & (patient.has_RB3 | patient.has_RB4)
patient["control1_new_sampling_eligible"] = patient.year.eq(2024) & patient.has_RB2
patient["control2_eligible"] = patient.year.eq(2024) & (patient.has_RB0 | patient.has_RB1)
patient["control3_eligible"] = patient.year.eq(2026) & (patient.has_RB3 | patient.has_RB4)

p24 = patient[patient.year.eq(2024)].copy()
p24["eligibility_pattern"] = p24[["target_eligible", "control1_new_sampling_eligible", "control2_eligible"]].apply(
    lambda r: "+".join([name for name, flag in zip(["target", "control1", "control2"], r) if flag]) or "none", axis=1
)
overlap_summary = p24.groupby("eligibility_pattern").agg(patients=("patient_id", "nunique"), images=("n_images", "sum")).reset_index()

rb_denominators = df.groupby(["year", "RB"]).agg(images=("image_key", "nunique"), patients=("patient_id", "nunique")).reset_index()
roi_long = df.melt(id_vars=["year", "patient_id", "image_key"], value_vars=ROIS, var_name="ROI", value_name="old_grade")
roi_population = roi_long[roi_long.old_grade.isin([3, 4])].groupby(["year", "ROI", "old_grade"]).agg(
    images=("image_key", "nunique"), patients=("patient_id", "nunique")
).reset_index()

print("환자 적격성")
display(patient)
print("2024년 대상군 중첩")
display(overlap_summary)
print("RB 기존 등급별 분모")
display(rb_denominators)
print("ROI별 기존 등급 3·4 분모")
display(roi_population)


환자 적격성


,year,patient_id,n_images,has_RB0,has_RB1,has_RB2,has_RB3,has_RB4,target_eligible,control1_new_sampling_eligible,control2_eligible,control3_eligible
0,2024,24_0407A,10,False,True,True,True,False,True,True,True,False
1,2024,24_0407B,13,False,True,True,False,False,False,True,True,False
2,2024,24_0407C,11,False,True,True,True,False,True,True,True,False
3,2024,24_0407D,9,False,True,True,True,False,True,True,True,False
4,2024,24_0407E,12,False,True,True,False,False,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
109,2026,26_0527A,13,True,False,False,True,False,False,False,False,True
110,2026,26_0527B,11,False,False,False,True,True,False,False,False,True
111,2026,26_0527D,7,True,True,True,False,False,False,False,False,False
112,2026,26_0528A,20,True,True,False,False,False,False,False,False,False


2024년 대상군 중첩


,eligibility_pattern,patients,images
0,control1,1,11
1,control1+control2,16,174
2,control2,6,62
3,target,5,50
4,target+control1,12,124
5,target+control1+control2,25,268
6,target+control2,3,29


RB 기존 등급별 분모


,year,RB,images,patients
0,2024,0,40,15
1,2024,1,263,50
2,2024,2,212,54
3,2024,3,135,42
4,2024,4,68,19
5,2026,0,87,23
6,2026,1,178,35
7,2026,2,145,33
8,2026,3,122,33
9,2026,4,55,16


ROI별 기존 등급 3·4 분모


,year,ROI,old_grade,images,patients
0,2024,LB,3,146,43
1,2024,LB,4,111,25
2,2024,LT,3,77,26
3,2024,LT,4,40,12
4,2024,RB,3,135,42
5,2024,RB,4,68,19
6,2024,RT,3,70,22
7,2024,RT,4,48,10
8,2026,LB,3,124,32
9,2026,LB,4,72,17


## 섹션 3. 대조군 표집량과 hidden duplicate 수 명시

**무엇을 하는가:** 섹션 2의 실제 count를 검토한 뒤 재판독에 보낼 규모를 직접 입력한다. 임의 기준으로 표집량을 자동 선택하지 않는다.

- `CONTROL1_PATIENT_CAP`: 2024년 RB 2를 가진 환자 중 신규 표집할 환자 수
- `CONTROL2_PATIENT_CAP`: 2024년 RB 0·1 원거리 대조군에서 표집할 환자 수
- `CONTROL3_MODE`: 2026년 RB 3·4를 `"census"` 또는 `"sample"`로 처리
- `CONTROL3_PATIENT_CAP`: `sample`일 때 표집할 환자 수
- `N_HIDDEN_DUPLICATES`: 최종 원본 mini-sequence가 만들어진 뒤 반복할 블록 수

대조군 문맥 창 길이는 계획서의 3–5장 범위에서 가능한 창을 모두 후보로 두고 확률표집한다.


In [9]:
CONTROL1_PATIENT_CAP = 10
CONTROL2_PATIENT_CAP = 10
CONTROL3_MODE = "census"   # "sample" 또는 "census"
CONTROL3_PATIENT_CAP = 10
N_HIDDEN_DUPLICATES = 10
CONTROL_WINDOW_LENGTHS = (3, 4, 5)

required = {
    "CONTROL1_PATIENT_CAP": CONTROL1_PATIENT_CAP,
    "CONTROL2_PATIENT_CAP": CONTROL2_PATIENT_CAP,
    "N_HIDDEN_DUPLICATES": N_HIDDEN_DUPLICATES,
}
if CONTROL3_MODE == "sample":
    required["CONTROL3_PATIENT_CAP"] = CONTROL3_PATIENT_CAP
if CONTROL3_MODE not in {"sample", "census"}:
    raise ValueError("CONTROL3_MODE은 'sample' 또는 'census'여야 합니다.")
if any(v is None for v in required.values()):
    raise ValueError(f"섹션 2의 count를 확인한 뒤 다음 값을 정수로 지정하세요: {[k for k,v in required.items() if v is None]}")


## 섹션 4. 대조군 확률표집과 영상별 포함확률

**무엇을 하는가:** 이동 가능성이 높은 구간을 골라내지 않고, 사전 정의된 확률표집으로 대조군 환자·중심 영상·문맥 창을 선택한다.

- 하위 경계 대조군: RB 2를 중심 영상으로 선택
- 원거리 음성 대조군: RB 0·1을 중심 영상으로 선택
- 원판독 조건 대조군: RB 3·4를 전수 또는 확률표집

원거리 대조군과 표본형 2026 대조군은 기존 등급 보유 패턴으로 환자 층을 만들고 층 내 단순무작위표집을 한다. 각 역할에 대해 환자 선택확률, 중심 영상 선택확률, 문맥 창 선택확률과 최종 영상 포함확률을 계산한다.


In [10]:
def grade_stratum(row, low, high):
    a, b = bool(row[f"has_RB{low}"]), bool(row[f"has_RB{high}"])
    if a and b: return f"{low}{high}_both"
    if a: return f"{low}_only"
    if b: return f"{high}_only"
    return "none"

def proportional_allocation(frame, n, stratum_col):
    counts = frame[stratum_col].value_counts().sort_index()
    if n > len(frame):
        raise ValueError(f"요청 환자 수 {n}가 후보 {len(frame)}명을 초과합니다.")
    raw_n = counts * n / counts.sum()
    alloc = np.floor(raw_n).astype(int)
    remainder = n - alloc.sum()
    if remainder:
        order = (raw_n - alloc).sort_values(ascending=False).index
        for stratum in order[:remainder]:
            alloc[stratum] += 1
    return alloc.to_dict()

def sample_patients(frame, n, stratum_col, seed):
    rng = np.random.default_rng(seed)
    alloc = proportional_allocation(frame, n, stratum_col)
    chosen, pi = [], {}
    for stratum, group in frame.groupby(stratum_col, sort=True):
        nh, Nh = alloc.get(stratum, 0), len(group)
        if nh == 0:
            continue
        ids = rng.choice(group.patient_id.to_numpy(), size=nh, replace=False)
        chosen.extend(ids.tolist())
        for pid in ids:
            pi[pid] = nh / Nh
    return chosen, pi, alloc

def candidate_windows(n_positions, center_pos, lengths=CONTROL_WINDOW_LENGTHS):
    windows = []
    for length in lengths:
        if length > n_positions:
            continue
        start_min = max(1, center_pos - length + 1)
        start_max = min(center_pos, n_positions - length + 1)
        for start in range(start_min, start_max + 1):
            windows.append(tuple(range(start, start + length)))
    return windows

def conditional_image_prob(patient_rows, eligible_grades):
    centers = patient_rows.loc[patient_rows.RB.isin(eligible_grades), "seq_position"].tolist()
    if not centers:
        return {}
    n = len(patient_rows)
    out = {pos: 0.0 for pos in patient_rows.seq_position}
    for center in centers:
        windows = candidate_windows(n, center)
        if not windows:
            continue
        for pos in out:
            out[pos] += (1/len(centers)) * sum(pos in w for w in windows) / len(windows)
    return out

def sample_group(role, candidate_patients, patient_cap, eligible_grades, seed, stratified=True):
    candidates = candidate_patients.copy()
    if stratified:
        low, high = min(eligible_grades), max(eligible_grades)
        candidates["sampling_stratum"] = candidates.apply(lambda r: grade_stratum(r, low, high), axis=1)
    else:
        candidates["sampling_stratum"] = "all"
    chosen_ids, patient_pi, allocation = sample_patients(candidates, patient_cap, "sampling_stratum", seed)
    rng = np.random.default_rng(seed + 1)
    selected_rows, meta_rows = [], []
    for pid in chosen_ids:
        rows = df[df.patient_id.eq(pid)].sort_values("seq_position").copy()
        centers = rows[rows.RB.isin(eligible_grades)].copy()
        if len(rows) < min(CONTROL_WINDOW_LENGTHS) or centers.empty:
            raise ValueError(f"{role}: 환자 {pid}에서 3–5장 문맥 창을 만들 수 없습니다.")
        center = centers.iloc[rng.integers(len(centers))]
        windows = candidate_windows(len(rows), int(center.seq_position))
        window = windows[rng.integers(len(windows))]
        marginal = conditional_image_prob(rows, eligible_grades)
        picked = rows[rows.seq_position.isin(window)].copy()
        picked[f"pi_{role}"] = picked.seq_position.map(lambda p: patient_pi[pid] * marginal[int(p)])
        selected_rows.append(picked)
        meta_rows.append({
            "role": role,
            "patient_id": pid,
            "sampling_stratum": candidates.set_index("patient_id").loc[pid, "sampling_stratum"],
            "patient_selection_probability": patient_pi[pid],
            "center_image_key": center.image_key,
            "center_old_grade": int(center.RB),
            "center_selection_probability": 1 / len(centers),
            "window_positions": ",".join(map(str, window)),
            "window_length": len(window),
            "window_selection_probability": 1 / len(windows),
        })
    return pd.concat(selected_rows, ignore_index=True), pd.DataFrame(meta_rows), allocation

p = patient.copy()
c1_candidates = p[p.control1_new_sampling_eligible].copy()
c2_candidates = p[p.control2_eligible].copy()
c3_candidates = p[p.control3_eligible].copy()

c1_rows, c1_meta, c1_alloc = sample_group("control1", c1_candidates, CONTROL1_PATIENT_CAP, {2}, RANDOM_SEED + 100, stratified=False)
c2_rows, c2_meta, c2_alloc = sample_group("control2", c2_candidates, CONTROL2_PATIENT_CAP, {0, 1}, RANDOM_SEED + 200, stratified=True)

if CONTROL3_MODE == "sample":
    c3_rows, c3_meta, c3_alloc = sample_group("control3", c3_candidates, CONTROL3_PATIENT_CAP, {3, 4}, RANDOM_SEED + 300, stratified=True)
else:
    c3_rows = df[(df.year.eq(2026)) & (df.RB.isin([3, 4]))].copy()
    c3_rows["pi_control3"] = 1.0
    c3_meta = pd.DataFrame([{"role": "control3", "mode": "census"}])
    c3_alloc = {"census": int(c3_candidates.patient_id.nunique())}

sampling_meta = pd.concat([c1_meta, c2_meta, c3_meta], ignore_index=True, sort=False)
display(sampling_meta)


,role,patient_id,sampling_stratum,patient_selection_probability,center_image_key,center_old_grade,center_selection_probability,window_positions,window_length,window_selection_probability,mode
0,control1,24_0601G,all,0.185185,24_0601G|4,2.0,0.200000,"3,4,5,6,7",5.0,0.090909,NaN
1,control1,24_0506B,all,0.185185,24_0506B|10,2.0,1.000000,"6,7,8,9,10",5.0,0.166667,NaN
2,control1,24_0407B,all,0.185185,24_0407B|7,2.0,0.200000,"5,6,7,8,9",5.0,0.083333,NaN
3,control1,24_0407J,all,0.185185,24_0407J|10,2.0,1.000000,"8,9,10",3.0,0.333333,NaN
4,control1,24_0506C,all,0.185185,24_0506C|9,2.0,0.333333,"6,7,8,9,10",5.0,0.166667,NaN
5,control1,24_1113D,all,0.185185,24_1113D|1,2.0,1.000000,"1,2,3,4",4.0,0.333333,NaN
6,control1,24_1213D,all,0.185185,24_1213D|7,2.0,0.333333,"5,6,7",3.0,0.083333,NaN
7,control1,24_0520G,all,0.185185,24_0520G|6,2.0,0.333333,"5,6,7,8",4.0,0.083333,NaN
8,control1,24_0506H,all,0.185185,24_0506H|6,2.0,0.333333,"6,7,8,9",4.0,0.083333,NaN
9,control1,24_0407I,all,0.185185,24_0407I|5,2.0,1.000000,"4,5,6,7",4.0,0.083333,NaN


## 섹션 5. 표적 전수와 대조군을 합친 mini-sequence 구성

**무엇을 하는가:** 2024 RB 3·4 표적 영상은 전수 포함하고 가능한 직전·직후 anchor를 붙인다. 대조군에서 선택된 문맥 창과 합친 뒤, 같은 영상이 여러 역할에 기여하면 한 번만 제시하면서 역할은 모두 기록한다.

환자 내부 인접성은 `seq` 숫자 차이가 아니라 `seq_position`으로 판단한다. 최종 선택 위치의 연속 구간마다 하나의 mini-sequence를 만든다.


In [11]:
def add_target_and_anchors(frame):
    selected = []
    for pid, rows in frame[frame.year.eq(2024)].groupby("patient_id"):
        rows = rows.sort_values("seq_position")
        core = set(rows.loc[rows.RB.isin([3, 4]), "seq_position"])
        if not core:
            continue
        positions = set(core)
        for pos in core:
            if pos > 1: positions.add(pos - 1)
            if pos < len(rows): positions.add(pos + 1)
        part = rows[rows.seq_position.isin(positions)].copy()
        part["pi_target"] = np.where(part.RB.isin([3, 4]), 1.0, np.nan)
        selected.append(part)
    return pd.concat(selected, ignore_index=True)

target_rows = add_target_and_anchors(df)
parts = []
for frame in [target_rows, c1_rows, c2_rows, c3_rows]:
    keep = ["image_key", "patient_id", "year", "seq", "seq_position", "patient_n_images", "source_image", *ROIS]
    pi_cols = [c for c in frame.columns if c.startswith("pi_")]
    parts.append(frame[keep + pi_cols])

selected = pd.concat(parts, ignore_index=True, sort=False)
agg = {c: "first" for c in ["patient_id", "year", "seq", "seq_position", "patient_n_images", "source_image", *ROIS]}
for c in ["pi_target", "pi_control1", "pi_control2", "pi_control3"]:
    if c in selected.columns:
        agg[c] = "max"
selected = selected.groupby("image_key", as_index=False).agg(agg)

for c in ["pi_target", "pi_control1", "pi_control2", "pi_control3"]:
    if c not in selected.columns:
        selected[c] = np.nan

selected["analysis_roles"] = selected.apply(lambda r: "|".join([
    role for role, cond in [
        ("target", pd.notna(r.pi_target) and r.RB in [3, 4]),
        ("control1", pd.notna(r.pi_control1) and r.RB == 2),
        ("control2", pd.notna(r.pi_control2) and r.RB in [0, 1]),
        ("control3", pd.notna(r.pi_control3) and r.RB in [3, 4]),
    ] if cond
]), axis=1)
selected["is_analysis_core"] = selected.analysis_roles.ne("")

blocks = []
for pid, rows in selected.groupby("patient_id"):
    rows = rows.sort_values("seq_position").copy()
    rows["new_block"] = rows.seq_position.diff().fillna(1).ne(1).astype(int)
    rows["component"] = rows.new_block.cumsum()
    for component, block in rows.groupby("component"):
        block = block.copy()
        block_id = f"{pid}|{int(component):02d}"
        block["mini_sequence_id"] = block_id
        block["mini_position"] = np.arange(1, len(block) + 1)
        block["minisequence_len"] = len(block)
        block["exposure_ratio"] = len(block) / int(block.patient_n_images.iloc[0])
        blocks.append(block)
mini = pd.concat(blocks, ignore_index=True)

analysis_core = mini[mini.is_analysis_core].copy()
mini_summary = mini.groupby(["patient_id", "mini_sequence_id"]).agg(
    sequence_length=("image_key", "size"),
    exposure_ratio=("exposure_ratio", "first"),
    anchor_or_context_images=("is_analysis_core", lambda s: int((~s).sum())),
    analysis_images=("is_analysis_core", "sum"),
).reset_index()

display(mini_summary)


,patient_id,mini_sequence_id,sequence_length,exposure_ratio,anchor_or_context_images,analysis_images
0,24_0407A,24_0407A|00,7,0.700000,4,3
1,24_0407B,24_0407B|00,8,0.615385,5,3
2,24_0407C,24_0407C|00,3,0.272727,2,1
3,24_0407D,24_0407D|00,3,0.333333,2,1
4,24_0407D,24_0407D|01,4,0.444444,2,2
...,...,...,...,...,...,...
108,26_0526B,26_0526B|00,2,0.142857,0,2
109,26_0527A,26_0527A|00,1,0.076923,0,1
110,26_0527B,26_0527B|00,11,1.000000,0,11
111,26_0528B,26_0528B|00,2,0.125000,0,2


## 섹션 6. 재판독 전 분모·ROI·가중치·문맥 감사

**무엇을 하는가:** 최종 mini-sequence에서 각 분석 역할의 실제 기존 등급 분모, 환자 수, 선택확률과 Kish 유효표본 수를 확인한다. 또한 RB 기준으로 선정된 2024 표적 영상 안에서 RT·RB·LT·LB의 기존 등급 3·4 분모를 센다.

문맥량 감사는 mini-sequence 길이 7장 이하, 전체 시퀀스 노출비율 0.8 이하, 두 조건 동시 적용에서 남는 분모를 표로 제시한다. 고정 통과율은 적용하지 않는다. 환자당 한 영상 분석과 문맥 보정 회귀에 필요한 환자·문맥 변수는 결과표에 보존한다.


In [12]:
role_specs = {
    "target": ("pi_target", {3, 4}),
    "control1": ("pi_control1", {2}),
    "control2": ("pi_control2", {0, 1}),
    "control3": ("pi_control3", {3, 4}),
}
role_rows = []
for role, (pi_col, grades) in role_specs.items():
    part = analysis_core[analysis_core.analysis_roles.str.contains(role, regex=False) & analysis_core.RB.isin(grades)].copy()
    for grade, g in part.groupby("RB"):
        w = 1 / g[pi_col]
        role_rows.append({
            "role": role,
            "old_grade": int(grade),
            "images": g.image_key.nunique(),
            "patients": g.patient_id.nunique(),
            "weight_sum": float(w.sum()),
            "kish_effective_n": float((w.sum() ** 2) / (w.pow(2).sum())),
            "min_inclusion_probability": float(g[pi_col].min()),
            "max_inclusion_probability": float(g[pi_col].max()),
        })
role_denominators = pd.DataFrame(role_rows)

target_image_keys = set(analysis_core.loc[analysis_core.analysis_roles.str.contains("target", regex=False), "image_key"])
target_roi = df[df.image_key.isin(target_image_keys)].melt(
    id_vars=["patient_id", "image_key"], value_vars=ROIS, var_name="ROI", value_name="old_grade"
)
roi_selected_denominators = target_roi[target_roi.old_grade.isin([3, 4])].groupby(["ROI", "old_grade"]).agg(
    images=("image_key", "nunique"), patients=("patient_id", "nunique")
).reset_index()

restrictions = {
    "all": pd.Series(True, index=analysis_core.index),
    "minisequence_len_le7": analysis_core.minisequence_len.le(7),
    "exposure_ratio_le0.8": analysis_core.exposure_ratio.le(0.8),
    "both": analysis_core.minisequence_len.le(7) & analysis_core.exposure_ratio.le(0.8),
}
survival = []
for name, mask in restrictions.items():
    sub = analysis_core[mask]
    for role, (_, grades) in role_specs.items():
        r = sub[sub.analysis_roles.str.contains(role, regex=False) & sub.RB.isin(grades)]
        for grade in sorted(grades):
            g = r[r.RB.eq(grade)]
            survival.append({"restriction": name, "role": role, "old_grade": grade,
                             "images": g.image_key.nunique(), "patients": g.patient_id.nunique()})
sensitivity_denominator_survival = pd.DataFrame(survival)

context_by_group = analysis_core.assign(role=analysis_core.analysis_roles.str.split("|")).explode("role").groupby("role").agg(
    images=("image_key", "nunique"), patients=("patient_id", "nunique"),
    median_sequence_length=("minisequence_len", "median"), median_exposure_ratio=("exposure_ratio", "median")
).reset_index()

print("분석 역할별 분모와 Kish 유효표본")
display(role_denominators)
print("RB 표적 영상에 조건부인 ROI 3·4 분모")
display(roi_selected_denominators)
print("문맥 제한별 분모 생존")
display(sensitivity_denominator_survival)
print("문맥량 요약")
display(context_by_group)


분석 역할별 분모와 Kish 유효표본


,role,old_grade,images,patients,weight_sum,kish_effective_n,min_inclusion_probability,max_inclusion_probability
0,target,3,135,42,135.000000,135.000000,1.000000,1.000000
1,target,4,68,19,68.000000,68.000000,1.000000,1.000000
2,control1,2,16,10,151.457609,14.190592,0.059671,0.185185
3,control2,0,1,1,8.679981,1.000000,0.115208,0.115208
4,control2,1,22,10,236.665102,17.302789,0.039583,0.200000
5,control3,3,122,33,122.000000,122.000000,1.000000,1.000000
6,control3,4,55,16,55.000000,55.000000,1.000000,1.000000


RB 표적 영상에 조건부인 ROI 3·4 분모


,ROI,old_grade,images,patients
0,LB,3,57,21
1,LB,4,58,18
2,LT,3,43,13
3,LT,4,13,6
4,RB,3,135,42
5,RB,4,68,19
6,RT,3,46,18
7,RT,4,42,8


문맥 제한별 분모 생존


,restriction,role,old_grade,images,patients
0,all,target,3,135,42
1,all,target,4,68,19
2,all,control1,2,16,10
3,all,control2,0,1,1
4,all,control2,1,22,10
5,all,control3,3,122,33
6,all,control3,4,55,16
7,minisequence_len_le7,target,3,64,28
8,minisequence_len_le7,target,4,11,6
9,minisequence_len_le7,control1,2,6,4


문맥량 요약


,role,images,patients,median_sequence_length,median_exposure_ratio
0,control1,16,10,8.0,0.707692
1,control2,23,10,5.0,0.444444
2,control3,177,33,7.0,0.562500
3,target,203,45,9.0,1.000000


## 섹션 7. Hidden duplicate 대상과 동일 문맥 반복 블록

**무엇을 하는가:** 최종 원본 mini-sequence 중 분석 core가 있는 블록을 후보로 두고, 사용자가 명시한 수만큼 블록을 단순무작위로 선택한다. 선택된 블록 안에서는 분석 core 영상 한 장을 균등하게 hidden duplicate 대상으로 정한다.

반복 블록은 원본과 같은 길이·같은 영상 순서를 그대로 복제한다. 대상 영상만 `hidden_duplicate_target`, 나머지는 `repeat_context`로 표시한다. 블록 선택확률, 블록 내 대상 선택확률과 두 확률의 곱을 기록한다.

실제 제시 순서는 원본 블록과 반복 블록을 함께 무작위화하고, 원본과 반복 블록 사이의 실제 간격을 결과표에 기록한다.


In [13]:
original = mini.copy()
original["occurrence_type"] = "original"
original["repeat_original_block_id"] = pd.NA

eligible_blocks = original[original.is_analysis_core].groupby("mini_sequence_id").agg(
    n_core=("is_analysis_core", "sum")
).reset_index()
if N_HIDDEN_DUPLICATES > len(eligible_blocks):
    raise ValueError(f"N_HIDDEN_DUPLICATES={N_HIDDEN_DUPLICATES}가 적격 블록 {len(eligible_blocks)}개를 초과합니다.")

rng_dup = np.random.default_rng(RANDOM_SEED + 400)
selected_blocks = rng_dup.choice(eligible_blocks.mini_sequence_id.to_numpy(), size=N_HIDDEN_DUPLICATES, replace=False)
block_pi = N_HIDDEN_DUPLICATES / len(eligible_blocks)

dup_targets, repeat_blocks = [], []
for i, block_id in enumerate(selected_blocks, 1):
    block = original[original.mini_sequence_id.eq(block_id)].copy()
    core = block[block.is_analysis_core]
    target = core.iloc[rng_dup.integers(len(core))]
    within_pi = 1 / len(core)
    duplicate_pi = block_pi * within_pi
    dup_targets.append({
        "original_mini_sequence_id": block_id,
        "target_image_key": target.image_key,
        "target_position": int(target.mini_position),
        "sequence_length": int(target.minisequence_len),
        "block_selection_probability": block_pi,
        "within_block_target_probability": within_pi,
        "duplicate_selection_probability": duplicate_pi,
    })
    repeat = block.copy()
    repeat["mini_sequence_id"] = f"REPEAT_{i:04d}"
    repeat["repeat_original_block_id"] = block_id
    repeat["occurrence_type"] = np.where(repeat.image_key.eq(target.image_key), "hidden_duplicate_target", "repeat_context")
    repeat["duplicate_selection_probability"] = duplicate_pi
    repeat_blocks.append(repeat)

duplicate_targets = pd.DataFrame(dup_targets)
repeats = pd.concat(repeat_blocks, ignore_index=True) if repeat_blocks else original.iloc[0:0].copy()

all_blocks = pd.concat([original, repeats], ignore_index=True, sort=False)
block_ids = all_blocks.mini_sequence_id.drop_duplicates().to_numpy()
rng_order = np.random.default_rng(RANDOM_SEED + 500)
ordered_blocks = rng_order.permutation(block_ids)
block_order = {block_id: i + 1 for i, block_id in enumerate(ordered_blocks)}
all_blocks["block_order"] = all_blocks.mini_sequence_id.map(block_order)
schedule = all_blocks.sort_values(["block_order", "mini_position"]).reset_index(drop=True)
schedule["read_order"] = np.arange(1, len(schedule) + 1)

original_block_order = {b: block_order[b] for b in original.mini_sequence_id.unique()}
repeat_info = []
for row in duplicate_targets.itertuples(index=False):
    repeat_id = repeats.loc[repeats.repeat_original_block_id.eq(row.original_mini_sequence_id), "mini_sequence_id"].iloc[0]
    repeat_info.append({
        "original_mini_sequence_id": row.original_mini_sequence_id,
        "repeat_mini_sequence_id": repeat_id,
        "original_block_order": original_block_order[row.original_mini_sequence_id],
        "repeat_block_order": block_order[repeat_id],
        "block_order_gap": abs(block_order[repeat_id] - original_block_order[row.original_mini_sequence_id]),
    })
duplicate_order_audit = pd.DataFrame(repeat_info)

display(duplicate_targets)
display(duplicate_order_audit)


,original_mini_sequence_id,target_image_key,target_position,sequence_length,block_selection_probability,within_block_target_probability,duplicate_selection_probability
0,26_0416B|00,26_0416B|7,2,2,0.088496,0.500000,0.044248
1,24_1213J|00,24_1213J|6,5,6,0.088496,0.500000,0.044248
2,24_0601D|01,24_0601D|10,2,2,0.088496,1.000000,0.088496
3,24_0506J|00,24_0506J|1,1,2,0.088496,1.000000,0.088496
4,26_0420A|00,26_0420A|4,4,6,0.088496,0.166667,0.014749
5,26_0427C|00,26_0427C|6,3,7,0.088496,0.142857,0.012642
6,24_0509H|00,24_0509H|4,4,11,0.088496,0.090909,0.008045
7,26_0508A|00,26_0508A|10,6,7,0.088496,0.142857,0.012642
8,26_0520C|00,26_0520C|5,5,12,0.088496,0.083333,0.007375
9,24_1113E|00,24_1113E|4,4,6,0.088496,0.200000,0.017699


,original_mini_sequence_id,repeat_mini_sequence_id,original_block_order,repeat_block_order,block_order_gap
0,26_0416B|00,REPEAT_0001,56,95,39
1,24_1213J|00,REPEAT_0002,29,43,14
2,24_0601D|01,REPEAT_0003,70,25,45
3,24_0506J|00,REPEAT_0004,17,13,4
4,26_0420A|00,REPEAT_0005,112,35,77
5,26_0427C|00,REPEAT_0006,68,107,39
6,24_0509H|00,REPEAT_0007,57,44,13
7,26_0508A|00,REPEAT_0008,1,36,35
8,26_0520C|00,REPEAT_0009,12,96,84
9,24_1113E|00,REPEAT_0010,62,92,30


## 섹션 8. 실제 재판독 제시 목록과 빈 반환 양식

**무엇을 하는가:** 무작위화된 최종 순서에 단순 case ID와 sequence ID를 부여한다. 재판독용 목록에는 기존 라벨, 촬영 연도, 환자 식별자, 대상군 역할과 중복 여부를 넣지 않는다.

원본 라벨과 분석 역할을 연결하는 표는 별도로 유지하며, 재판독 결과가 반환된 뒤 `original`, `hidden_duplicate_target`, `repeat_context`를 구분하는 데 사용한다.


In [14]:
schedule = schedule.sort_values("read_order").reset_index(drop=True)
schedule["case_id"] = [f"P2A_C{i:06d}" for i in range(1, len(schedule) + 1)]
sequence_ids = {b: f"P2A_S{i:04d}" for i, b in enumerate(schedule.mini_sequence_id.drop_duplicates(), 1)}
schedule["reread_sequence_id"] = schedule.mini_sequence_id.map(sequence_ids)
schedule["session_id"] = pd.NA

reread_manifest = schedule[[
    "read_order", "reread_sequence_id", "mini_position", "minisequence_len", "case_id"
]].rename(columns={"mini_position": "sequence_position", "minisequence_len": "sequence_length"})

reread_template = reread_manifest[["read_order", "reread_sequence_id", "sequence_position", "case_id"]].merge(
    pd.DataFrame({"ROI": ROIS}), how="cross"
)
reread_template["new_grade"] = pd.NA
reread_template["evaluable"] = pd.NA
reread_template["non_evaluable_reason"] = pd.NA
reread_template["reread_timestamp"] = pd.NA

selection_key = schedule[[
    "case_id", "read_order", "reread_sequence_id", "mini_sequence_id", "mini_position", "minisequence_len",
    "image_key", "patient_id", "year", "seq", "seq_position", "source_image", "occurrence_type",
    "repeat_original_block_id", "analysis_roles", "pi_target", "pi_control1", "pi_control2", "pi_control3",
    "RT", "RB", "LT", "LB"
]].copy()

forbidden = {"patient_id", "year", "analysis_roles", "occurrence_type", "RT", "RB", "LT", "LB"}
assert forbidden.isdisjoint(reread_manifest.columns)

display(reread_manifest.head(20))
display(reread_template.head(12))


,read_order,reread_sequence_id,sequence_position,sequence_length,case_id
0,1,P2A_S0001,1,7,P2A_C000001
1,2,P2A_S0001,2,7,P2A_C000002
2,3,P2A_S0001,3,7,P2A_C000003
3,4,P2A_S0001,4,7,P2A_C000004
4,5,P2A_S0001,5,7,P2A_C000005
5,6,P2A_S0001,6,7,P2A_C000006
6,7,P2A_S0001,7,7,P2A_C000007
7,8,P2A_S0002,1,6,P2A_C000008
8,9,P2A_S0002,2,6,P2A_C000009
9,10,P2A_S0002,3,6,P2A_C000010


,read_order,reread_sequence_id,sequence_position,case_id,ROI,new_grade,evaluable,non_evaluable_reason,reread_timestamp
0,1,P2A_S0001,1,P2A_C000001,RT,<NA>,<NA>,<NA>,<NA>
1,1,P2A_S0001,1,P2A_C000001,RB,<NA>,<NA>,<NA>,<NA>
2,1,P2A_S0001,1,P2A_C000001,LT,<NA>,<NA>,<NA>,<NA>
3,1,P2A_S0001,1,P2A_C000001,LB,<NA>,<NA>,<NA>,<NA>
4,2,P2A_S0001,2,P2A_C000002,RT,<NA>,<NA>,<NA>,<NA>
5,2,P2A_S0001,2,P2A_C000002,RB,<NA>,<NA>,<NA>,<NA>
6,2,P2A_S0001,2,P2A_C000002,LT,<NA>,<NA>,<NA>,<NA>
7,2,P2A_S0001,2,P2A_C000002,LB,<NA>,<NA>,<NA>,<NA>
8,3,P2A_S0001,3,P2A_C000003,RT,<NA>,<NA>,<NA>,<NA>
9,3,P2A_S0001,3,P2A_C000003,RB,<NA>,<NA>,<NA>,<NA>


## 섹션 9. 재판독 영상 선정 단계 산출물 저장

**무엇을 하는가:** 계획서 10장의 산출물에 대응하는 CSV를 저장한다. 이 섹션은 영상 선정 결과만 저장하며 재판독 후 통계 분석을 실행하지 않는다.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "patient_eligibility.csv": patient,
    "patient_overlap_2024.csv": p24,
    "patient_overlap_summary.csv": overlap_summary,
    "rb_grade_denominators.csv": rb_denominators,
    "roi_population_denominators.csv": roi_population,
    "sampling_window_metadata.csv": sampling_meta,
    "analysis_image_list.csv": analysis_core,
    "selection_probabilities.csv": role_denominators,
    "minisequence_images.csv": mini,
    "minisequence_summary.csv": mini_summary,
    "roi_selected_set_denominators.csv": roi_selected_denominators,
    "context_denominator_survival.csv": sensitivity_denominator_survival,
    "context_by_group.csv": context_by_group,
    "hidden_duplicate_targets.csv": duplicate_targets,
    "hidden_duplicate_blocks.csv": repeats,
    "duplicate_order_audit.csv": duplicate_order_audit,
    "presentation_order.csv": schedule,
    "selection_key.csv": selection_key,
    "reread_manifest.csv": reread_manifest,
    "reread_template.csv": reread_template,
}
for name, table in outputs.items():
    table.to_csv(OUTPUT_DIR / name, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_DIR}")
display(pd.DataFrame([{"file": name, "rows": len(table)} for name, table in outputs.items()]))
